##Read the three Bronze tables

In [0]:
from pyspark.sql.functions import col, trim, upper, expr, lit
from pyspark.sql import functions as F

product_bronze = spark.table(
    "capretailnew.bronze.bronze_product_master"
)

store_bronze = spark.table(
    "capretailnew.bronze.bronze_store_master"
)

supplier_bronze = spark.table(
    "capretailnew.bronze.bronze_supplier_master"
)

print("Product:", product_bronze.count())
print("Store:", store_bronze.count())
print("Supplier:", supplier_bronze.count())

##Transform Product Master

In [0]:
product_silver = (
    product_bronze
    .select(
        trim(col("product_id")).alias("product_id"),
        trim(col("sku")).alias("sku"),
        trim(col("product_name")).alias("product_name"),
        trim(col("category")).alias("category"),
        trim(col("subcategory")).alias("subcategory"),
        trim(col("unit_of_measure")).alias("unit_of_measure"),
        expr("try_cast(trim(list_price) AS DECIMAL(18,2))").alias("list_price"),
        expr("try_cast(trim(cost_price) AS DECIMAL(18,2))").alias("cost_price"),
        trim(col("supplier_id")).alias("supplier_id"),
        upper(trim(col("active_flag"))).alias("active_flag")
    )
)

##Transform Store Master

In [0]:
store_silver = (
    store_bronze
    .select(
        trim(col("store_id")).alias("store_id"),
        trim(col("store_name")).alias("store_name"),
        trim(col("city")).alias("city"),
        trim(col("region")).alias("region"),
        trim(col("store_type")).alias("store_type"),
        expr("try_cast(trim(opening_date) AS DATE)").alias("opening_date"),
        upper(trim(col("active_flag"))).alias("active_flag"),
        trim(col("warehouse_id")).alias("warehouse_id"),
        trim(col("manager_code")).alias("manager_code"),
        trim(col("country")).alias("country")
    )
)

##Transform Supplier Master

In [0]:
supplier_silver = (
    supplier_bronze
    .select(
        trim(col("supplier_id")).alias("supplier_id"),
        trim(col("supplier_name")).alias("supplier_name"),
        trim(col("country")).alias("country"),
        expr("try_cast(trim(lead_time_days) AS INT)").alias("lead_time_days"),
        trim(col("payment_terms")).alias("payment_terms"),
        upper(trim(col("active_flag"))).alias("active_flag"),
        trim(col("supplier_tier")).alias("supplier_tier"),
        trim(col("city")).alias("city"),
        trim(col("category")).alias("category"),
        trim(col("contact_code")).alias("contact_code")
    )
)

##Validate casting and primary keys

In [0]:
def validate_master(df, key, cast_columns):
    total = df.count()

    missing_keys = df.filter(
        col(key).isNull() | (trim(col(key)) == "")
    ).count()

    duplicate_keys = (
        df.groupBy(key)
        .count()
        .filter(col("count") > 1)
        .count()
    )

    print("\nTable:", key)
    print("Total records:", total)
    print("Missing keys:", missing_keys)
    print("Duplicate key groups:", duplicate_keys)

    for column_name in cast_columns:
        invalid = df.filter(col(column_name).isNull()).count()
        print("NULL / failed cast in", column_name, ":", invalid)


validate_master(
    product_silver,
    "product_id",
    ["list_price", "cost_price"]
)

validate_master(
    store_silver,
    "store_id",
    ["opening_date"]
)

validate_master(
    supplier_silver,
    "supplier_id",
    ["lead_time_days"]
)

##Validate product-to-supplier relationship

In [0]:
supplier_ids = (
    supplier_silver
    .select("supplier_id")
    .distinct()
    .withColumn("supplier_exists", lit(True))
)

product_supplier_validation = product_silver.join(
    supplier_ids,
    on="supplier_id",
    how="left"
)

display(
    product_supplier_validation
    .groupBy("supplier_exists")
    .count()
)

##Save the three Silver master tables


In [0]:
(product_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("capretailnew.silver.silver_product_master")
)

(store_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("capretailnew.silver.silver_store_master")
)

(supplier_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("capretailnew.silver.silver_supplier_master")
)

print("All three Silver master tables created.")

##Silver Layer Validation

In [0]:
sales = spark.table("capretailnew.silver.silver_sales_order_lines")
inventory = spark.table("capretailnew.silver.silver_inventory_movements")
returns = spark.table("capretailnew.silver.silver_sales_returns")

products = spark.table("capretailnew.silver.silver_product_master")
stores = spark.table("capretailnew.silver.silver_store_master")
suppliers = spark.table("capretailnew.silver.silver_supplier_master")

print("All six Silver tables loaded.")
table_counts = [
    ("Sales Silver", sales.count(), 11320),
    ("Sales Quarantine", spark.table(
        "capretailnew.silver.quarantine_sales_order_lines"
    ).count(), 53),
    ("Sales Duplicate Audit", spark.table(
        "capretailnew.silver.sales_order_lines_duplicate_audit"
    ).count(), 39),
    ("Inventory Silver", inventory.count(), 634),
    ("Inventory Quarantine", spark.table(
        "capretailnew.silver.quarantine_inventory_movements"
    ).count(), 0),
    ("Returns Silver", returns.count(), 316),
    ("Returns Quarantine", spark.table(
        "capretailnew.silver.quarantine_sales_returns"
    ).count(), 0),
    ("Product Silver", products.count(), 139),
    ("Store Silver", stores.count(), 148),
    ("Supplier Silver", suppliers.count(), 157)
]

for name, actual, expected in table_counts:
    print(
        name,
        "| Actual:", actual,
        "| Expected:", expected,
        "| Status:", "PASS" if actual == expected else "CHECK"
    )
    primary_keys = [
    (sales, "order_line_id", "Sales"),
    (inventory, "movement_id", "Inventory"),
    (returns, "return_id", "Returns"),
    (products, "product_id", "Product"),
    (stores, "store_id", "Store"),
    (suppliers, "supplier_id", "Supplier")
]

for df, key, name in primary_keys:
    duplicate_count = (
        df.groupBy(key)
        .count()
        .filter(col("count") > 1)
        .count()
    )

    missing_count = df.filter(
        col(key).isNull() | (trim(col(key)) == "")
    ).count()

    print(
        name,
        "| Duplicate key groups:", duplicate_count,
        "| Missing keys:", missing_count
    )
    

##Validate foreign-key relationships

In [0]:
def check_orphans(source_df, source_column, target_df, target_column, label):
    target_keys = (
        target_df.select(col(target_column).alias("_target_key"))
        .distinct()
    )

    orphan_count = (
        source_df
        .join(
            target_keys,
            source_df[source_column] == target_keys["_target_key"],
            "left_anti"
        )
        .filter(
            col(source_column).isNotNull() &
            (trim(col(source_column)) != "")
        )
        .count()
    )

    print(label, "| Orphan records:", orphan_count)


# Sales relationships
check_orphans(sales, "product_id", products, "product_id", "Sales -> Product")
check_orphans(sales, "store_id", stores, "store_id", "Sales -> Store")

# Inventory relationships
check_orphans(inventory, "product_id", products, "product_id", "Inventory -> Product")
check_orphans(inventory, "store_id", stores, "store_id", "Inventory -> Store")

# Returns relationships
check_orphans(returns, "product_id", products, "product_id", "Returns -> Product")
check_orphans(returns, "store_id", stores, "store_id", "Returns -> Store")

# Product master relationship
check_orphans(products, "supplier_id", suppliers, "supplier_id", "Product -> Supplier")

##Validate Returns-to-Sales relationship

In [0]:
sales_quarantine = spark.table(
    "capretailnew.silver.quarantine_sales_order_lines"
)

clean_sales_ids = sales.select("order_line_id").distinct()
quarantined_sales_ids = sales_quarantine.select("order_line_id").distinct()

return_clean_match = returns.join(
    clean_sales_ids,
    on="order_line_id",
    how="left_semi"
).count()

return_quarantine_match = returns.join(
    quarantined_sales_ids,
    on="order_line_id",
    how="left_semi"
).count()

print("Returns linked to clean Sales:", return_clean_match)
print("Returns linked to quarantined Sales:", return_quarantine_match)
print("Total accounted returns:", return_clean_match + return_quarantine_match)

##Review chronology flags

In [0]:
display(
    returns.groupBy(
        "sales_link_status",
        "return_before_order"
    )
    .count()
    .orderBy("sales_link_status", "return_before_order")
)